# Recalibración CO₂ v3 — datos crudos completos y filtrado offline sin retardo de fase

Esta versión rehace la recalibración de LAB016–021 desde `flow_sccm`, conserva los notebooks anteriores y escribe únicamente tablas independientes junto a este notebook. **Todas las figuras quedan en los outputs: ninguna se guarda en `results`.**

**Esta ejecución corrige el cambio de hora de Chile y elimina las exclusiones de la primera v3.** La respuesta a la nutrición se conserva en los datos y en el ajuste.

**Cambios frente a v2:** observaciones cada **10 minutos** en vez de una hora; CO₂ y temperatura de los backups **completos** de LAB019–021; Hampel centrado seguido de Butterworth adelante–atrás desde los datos crudos; temperatura suavizada y utilizada efectivamente tanto por la química como por el gas; horizontes auditados hasta CO₂ cercano a cero y última química. Los puntos del ajuste son medianas de ventanas de 10 minutos situadas en su centro; no se desplazan registros 40 minutos ni se ajusta un retardo empírico.

**Configuración principal fijada antes de ajustar:** CO₂ con Butterworth orden 3, τ=30 min, dos pasadas; temperatura con mediana de 1 min → mediana móvil centrada de 20 min → media móvil centrada de 45 min. Se contrastan τ=20 y 40 min y temperatura 30/60 min. No se elige un filtro por obtener el menor RMSE sobre su propia señal suavizada: se publican también errores frente a una referencia común derivada del raw sin Butterworth.

Los parámetros químicos `theta_full` y `theta_defaults` se mantienen fijos. La fuente principal conserva las ecuaciones de v2 y sus posibles limitaciones; las mezclas positivas se presentan al final como una prueba estructural aparte. **Entrenamiento principal: LAB019–021; transferencia: LAB016–018.** También se conserva el ajuste conjunto descriptivo y la transferencia inversa de v2, sin presentarlos como validación externa. LAB016–018 ya se ha inspeccionado: toda esta comparación es exploratoria.

La inicialización heredada de LAB016–018 sigue siendo X₀=0.39 g/L, Xd₀=0 y G₀+F₀=172.5 g/L. Las primeras muestras de biomasa a ~10–11.5 h contradicen parte del crecimiento predicho; se muestran explícitamente y no se convierten silenciosamente en mediciones a t=0. Esta tarea cambia el tratamiento de señales, no reestima esa inicialización ni las energías de activación químicas.

In [ ]:
from pathlib import Path
from datetime import datetime
import ast, json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import display, Markdown
%matplotlib inline

ROOT=Path.cwd().resolve()
while ROOT.name!='pyomo-doe' and ROOT.parent!=ROOT:ROOT=ROOT.parent
assert ROOT.name=='pyomo-doe'
ND=ROOT/'fermentation_model/laboratory_2026/notebooks/diagnostics'
sys.path.insert(0,str(ND))
from _co2_recalibration_v3 import (RecalibrationV3,LABS,FILTERS,PRIMARY,OBS_STEP_H,FACTOR,
    centered_hampel,zero_phase,butter_sos,sha256,local_to_utc,LOCAL_ZONE)
assert Path(sys.prefix).name.lower()=='fermentation',sys.executable
OUT=ND/'co2_lab016_021_recalibration_v3_artifacts'
OUT.mkdir(exist_ok=True)
START=time.perf_counter()
V=RecalibrationV3(ROOT)
pd.set_option('display.max_columns',30)
pd.set_option('display.max_rows',70)
print('Python:',sys.executable)
display(pd.DataFrame([dict(file=p.name,**V.gui) for p in V.gui_paths]))
FIGURE_COUNT=0
def show_v3(fig,legend=False):
    global FIGURE_COUNT
    title=fig._suptitle is not None
    if title:fig._suptitle.set_y(.995)
    for item in fig.legends:item.set_bbox_to_anchor((.5,.95 if title else .995),transform=fig.transFigure)
    fig.tight_layout(rect=(0,0,1,.87 if title else .93 if legend else .97))
    display(fig);plt.close(fig);FIGURE_COUNT+=1

## 1. Qué significa aplicar el filtro de la GUI de forma offline

Se leen los parámetros de los dos scripts de `context/codigo_GUI` mediante AST, sin importar ni ejecutar su interfaz o hardware. Ambos coinciden: Hampel de 120 s, k=3.5, mínimo de pico 0.5 SCCM; Butterworth orden 3, τ=1200 s, f_c=1/(2πτ). La GUI ejecuta **Hampel → Butterworth causal**, no en el orden inverso.

Aquí la ventana Hampel se centra y se hace impar para ser simétrica; conserva el criterio exacto mediana/MAD de una misma ventana. El Butterworth usa los mismos coeficientes de diseño, pero se aplica con `sosfiltfilt` adelante–atrás. Esto elimina el retardo de fase del filtro lineal; **también eleva al cuadrado su respuesta de amplitud** (orden efectivo duplicado). τ=30 min baja además la frecuencia de corte respecto a los 20 min de la GUI. Véase [SciPy: sosfiltfilt](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.sosfiltfilt.html).

Es un procesamiento offline que utiliza muestras futuras; no se propone como sustituto causal para operación en vivo. Sin fase no significa que cualquier máximo o cruce de umbral conserve exactamente su hora: suavizar puede cambiar cuál de dos máximos cercanos es el mayor. Se conservan los timestamps civiles originales como trazabilidad; los cálculos se realizan en UTC y las figuras muestran horas realmente transcurridas.

**Corrección de esta v3:** se retiran las máscaras por activación de bomba, los 90 minutos de recuperación y todos los márgenes de exclusión de cuatro horas, incluidos los bordes. La activación corresponde a nutrición: se conserva la señal antes, durante y después, sin puentes lineales sobre esos eventos. El Hampel sólo sustituye desviaciones puntuales según su ventana de 120 segundos; una respuesta sostenida no se elimina por coincidir con la bomba. Esto no garantiza distinguir toda anomalía instrumental de una respuesta real: se muestran el raw despikeado y la sensibilidad del filtro.

**Cambio de hora de Chile:** los timestamps civiles se interpretan con `America/Santiago` y se convierten a UTC **antes** de remuestrear, filtrar o restar la inoculación. El salto local de 05/09/2026 23:59:47 a 06/09/2026 01:00:01 equivale a sólo **14 segundos reales**, no a una hora sin registros. CO₂, temperatura, tiempos de muestras y nutriciones comparten el mismo eje transcurrido. Los valores de química, dosis e iniciales permanecen iguales; se conserva el mapeo explícito de muestras iniciales a t=0. La auditoría exporta ambos relojes. Los tiempos anteriores al cambio no se desplazan y los posteriores se reducen una hora respecto al cálculo civil de v2.

No se añaden márgenes alrededor de huecos o extremos. Sólo una ausencia real de registros mayor de un minuto separa bloques de filtrado; los pequeños huecos de la grilla regular (≤60 s) mantienen la interpolación local documentada. El padding impar del filtro es una extensión numérica, **no una exclusión de datos**. Los extremos se conservan y tienen mayor incertidumbre de filtrado.

La temperatura replica el suavizado centrado 1/20/45 min mostrado en `lab019_021_co2_onset_effective_model_test`. En aquel notebook, `temperature_profile` usaba T cruda aunque las figuras mostraban T filtrada. Aquí se corrige esa discordancia: la misma T filtrada alimenta cinética, solubilidad, O₂ y figuras.

In [ ]:
V.process()
V.coverage.to_csv(OUT/'coverage.csv',index=False)
V.filter_audit.to_csv(OUT/'filter_audit.csv',index=False)
V.temp_audit.to_csv(OUT/'temperature_filter_audit.csv',index=False)
display(V.coverage.round(4))
display(V.filter_audit[['lab','variant','tau_s','raw_dt_s','zero_sccm','n_hampel','peak_g_l_h','peak_h','guard_h']].round(5))
display(V.obs.groupby('lab').agg(n_10min=('y','size'),first_fit_h=('t_h','min'),last_fit_h=('t_h','max')))
display(V.v2_obs.groupby('lab').size().rename('n_v2_hourly').to_frame())
print('Cada variante usa los mismos intervalos válidos de 10 minutos; sin márgenes ni máscaras por nutrición.')
CLOCK_AUDIT=pd.DataFrame(V.clock_audit)
display(CLOCK_AUDIT)
CLOCK_AUDIT.to_csv(OUT/'clock_change_audit.csv',index=False)
pd.DataFrame(V.elapsed_audit).to_csv(OUT/'elapsed_time_audit.csv',index=False)
V.initials.reset_index().to_csv(OUT/'initial_conditions_corrected_clock.csv',index=False)
assert len(CLOCK_AUDIT)==6
assert (CLOCK_AUDIT.elapsed_interval_seconds<30.).all()
assert (CLOCK_AUDIT.civil_interval_seconds-CLOCK_AUDIT.elapsed_interval_seconds==3600.).all()
clock=local_to_utc(['2026-09-05 23:59:47','2026-09-06 01:00:01'])
assert (clock[1]-clock[0]).total_seconds()==14.
NUTRITION_SUPPORT=[]
for lab in LABS:
    d=V.dense[lab]
    assert str(d.index.tz)=='UTC' and not V.fit_exclusions[lab] and not V.artifacts[lab]
    # Every finite filtered sample is retained, including edges and nutrition.
    assert np.array_equal(d.valid_for_fit.to_numpy(),d[PRIMARY].notna().to_numpy())
    if lab in LABS[:3]:
        # No actual logger gap at the daylight-saving transition.
        raw=V.raw_co2[lab]
        assert raw.timestamp.diff().dt.total_seconds().max()<60.
        assert d.loc[d.time_h.between(115.,125.),'valid_for_fit'].all()
    for t,dose in V.batches[lab].pulses['N']:
        if t<=0:continue
        support=d.time_h.between(t-1.,t+2.)
        obs=V.observations[PRIMARY,lab]
        n=int(obs.t_h.between(t-1.,t+2.).sum())
        assert d.loc[support,'valid_for_fit'].all() and n>=17
        NUTRITION_SUPPORT.append(dict(lab=lab,nutrition_h=t,dose_N=dose,
            valid_dense_fraction=float(d.loc[support,'valid_for_fit'].mean()),
            observations_in_minus1_plus2h=n))
NUTRITION_SUPPORT=pd.DataFrame(NUTRITION_SUPPORT)
display(NUTRITION_SUPPORT)
NUTRITION_SUPPORT.to_csv(OUT/'nutrition_support_checks.csv',index=False)

## 2. Cobertura completa, cortes y extremos que no se pueden confirmar

Se lee el final real de ambos backups completos. El corte por LAB es el más tardío entre la última muestra química/offline y la confirmación de dos horas continuas con CO₂≤0.02 g/L/h, después del último tramo detectable ≥0.05. Si no se puede confirmar el apagado con soporte válido, se conserva el final disponible y se etiqueta **right_censored**; no se fuerza cero ni se inventa una cola. Las muestras NUT con horario no confirmado se dibujan como contexto, pero no fijan el final ni los errores.

V2 ya había leído los backups completos, pero cortaba LAB019–021 a las 166 h de la última química. Ahora se comprueba también la terminación del CO₂: **LAB020 requiere prolongar la observación hasta ~172.3 h** con la configuración principal. LAB019 y LAB021 mantienen al menos la química de 166 h. El contexto posterior al corte se utiliza para filtrar sin crear un borde artificial; no se incorpora automáticamente todo el logger al objetivo.

El cero instrumental se estima una sola vez por LAB sobre el raw despikeado entre 2.5 y 12 h, fuera de artefactos, con el percentil 10 truncado a cero. Se comparte entre los tres filtros; puede diferir del cero anterior estimado sobre la señal causal. El inventario incluye la diferencia para distinguirla del cambio de intensidad del filtro.

In [ ]:
CO2_MINUTE=[];TEMP_EXPORT=[];ARTIFACT_ROWS=[]
for cohort in (LABS[3:],LABS[:3]):
    fig,axs=plt.subplots(3,1,figsize=(15,11),sharex=True)
    for ax,lab in zip(axs,cohort):
        d=V.dense[lab];cut=V.horizons[lab]
        t=d.time_h.to_numpy();keep=(t>=0.)&(t<=cut)
        raw=np.where(d.valid_for_fit,d.despiked,np.nan)
        ax.plot(t[keep],raw[keep],c='0.6',lw=.65,alpha=.65,label='Raw despikeado · soporte válido')
        ax.plot(t[keep],d.recorded.to_numpy()[keep],c='0.25',ls='--',lw=.9,alpha=.7,label='Filtrado causal guardado (referencia)')
        for filtering,color in [('offline_20min','#56B4E9'),(PRIMARY,'#D55E00'),('offline_40min','#009E73')]:
            values=np.where(d.valid_for_fit,d[filtering],np.nan)
            ax.plot(t[keep],values[keep],c=color,lw=1.8 if filtering==PRIMARY else 1.,label=filtering)
        temp=V.temperature[lab]
        tx=ax.twinx();tx.set_zorder(ax.get_zorder()-1);ax.patch.set_visible(False)
        tx.plot(temp.time_h,temp.reference_20_45,c='#9273B2',alpha=.5,lw=1.2,label='T filtrada 20/45 min')
        tx.set_ylabel('Temperatura [°C]',color='#795397');tx.tick_params(axis='y',colors='#795397')
        for event_h,dose in V.batches[lab].pulses['N']:
            ax.axvline(event_h,c='deeppink',ls=':',lw=.9,label='Nutrición · respuesta conservada' if event_h==V.batches[lab].pulses['N'][0][0] else None)
        ax.axvline(float(V.coverage.set_index('lab').loc[lab,'last_chemical_or_offline_h']),c='purple',ls=':',lw=1.,label='Última química/offline')
        ax.set(title=f'{lab} · raw y filtros offline; corte={cut:.2f} h',ylabel='g CO₂/L/h',xlim=(0.,max(V.horizons[l] for l in cohort)))
        ax.grid(alpha=.2)
    axs[-1].set_xlabel('Tiempo desde inoculación [h]')
    handles,labels=axs[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='upper center',ncol=3,fontsize=9)
    show_v3(fig,legend=True)

fig,axs=plt.subplots(2,3,figsize=(17,7),sharex=True)
for ax,lab in zip(axs.flat,LABS):
    temp=V.temperature[lab];keep=temp.time_h.between(0.,V.horizons[lab])
    ax.plot(temp.time_h[keep],temp.raw_1min[keep],c='0.65',lw=.7,label='T medida · mediana 1 min')
    ax.plot(temp.time_h[keep],temp.reference_20_45[keep],c='#D55E00',lw=1.7,label='20 min mediana + 45 min media')
    ax.plot(temp.time_h[keep],temp.strong_30_60[keep],c='#0072B2',lw=1.3,ls='--',label='30 min mediana + 60 min media')
    ax.set(title=lab,ylabel='°C',xlabel='Tiempo desde inoculación [h]');ax.grid(alpha=.2)
fig.legend(*axs[0,0].get_legend_handles_labels(),loc='upper center',ncol=3,fontsize=9)
show_v3(fig,legend=True)
display(V.temp_audit.round(5))
for lab in LABS:
    d=V.dense[lab].copy()
    CO2_MINUTE.append(d.resample('1min').median(numeric_only=True).assign(lab=lab).reset_index(names='timestamp'))
    TEMP_EXPORT.append(V.temperature[lab].assign(lab=lab).reset_index(names='timestamp'))
    ARTIFACT_ROWS.extend(dict(lab=lab,**event) for event in V.artifacts[lab])
pd.concat(CO2_MINUTE,ignore_index=True).to_csv(OUT/'preprocessed_co2_1min_preview.csv',index=False)
pd.concat(TEMP_EXPORT,ignore_index=True).to_csv(OUT/'temperature_inputs_1min.csv',index=False)
pd.DataFrame(ARTIFACT_ROWS,columns=['lab','start_h','end_h']).to_csv(OUT/'artifact_windows.csv',index=False)
pd.DataFrame(V.nutrition_events).to_csv(OUT/'nutrition_events.csv',index=False)

## 3. Comprobaciones del filtrado: fase, picos y bordes

Una señal gaussiana simétrica permite separar el retardo del filtro de los cambios de forma en una curva real. Se comprueba el máximo, el punto medio de un escalón y que el Hampel retire un outlier aislado sin desplazar los vecinos constantes. El filtro causal se incluye únicamente como control del orden de magnitud del retardo, no para desplazar observaciones.

Se conserva padding impar de seis constantes de tiempo como extensión numérica; no se excluyen los bordes del ajuste. Al suavizar una señal real con varios máximos próximos, el máximo global puede cambiar de uno a otro: las diferencias de hora de la tabla de filtros no son automáticamente un retardo de fase.

In [ ]:
FILTER_CHECKS=[]
dt=10.;tt=np.arange(-12*3600.,12*3600.+dt,dt)
gaussian=np.exp(-.5*(tt/(2.*3600.))**2)
step=(tt>=0.).astype(float)
for name,tau in FILTERS.items():
    z=zero_phase(gaussian,tau,dt)
    ss=zero_phase(step,tau,dt)
    peak_shift=float(tt[np.argmax(z)])
    midpoint=float(tt[np.flatnonzero(ss>=.5)[0]])
    assert abs(peak_shift)<=dt and abs(midpoint)<=dt
    FILTER_CHECKS.append(dict(test=name,peak_shift_seconds=peak_shift,step_midpoint_seconds=midpoint,
        amplitude_ratio=float(z.max()/gaussian.max())))
test=np.ones(301);test[150]=50.
despiked,flags=centered_hampel(test,10.)
assert flags.sum()==1 and flags[150] and np.array_equal(despiked,np.ones(301))
causal=signal.sosfilt(butter_sos(1200.,dt),gaussian)
FILTER_CHECKS.append(dict(test='GUI Butterworth causal control',peak_shift_seconds=float(tt[np.argmax(causal)]),
    step_midpoint_seconds=np.nan,amplitude_ratio=float(causal.max())))
FILTER_CHECKS=pd.DataFrame(FILTER_CHECKS)
display(FILTER_CHECKS)
FILTER_CHECKS.to_csv(OUT/'filter_synthetic_checks.csv',index=False)

## 4. Química con la temperatura filtrada y coherencia del modelo de observación

Se integran las ecuaciones químicas oficiales con saltos de N exactos. La malla de salida de 5 min es más fina que los datos del ajuste. Se repiten las simulaciones con T 20/45, T 30/60 y T con sólo mediana de 1 min para medir cuánto cambia la química al suavizar la temperatura; no se ajusta ningún theta.

La fuente de v2 sigue siendo q_bio=M·max[(1+(b−1)r)q₀+r(qₙ−q₀),0], con puerta por consumo predicho de G+F entre 2 y 10 g/L, b=0.25 y los demás parámetros heredados. El reservorio se integra de forma implícita conservativa. **La predicción que se compara con la observación pasa por el mismo Butterworth offline que los datos**, en la grilla nativa del logger. Esto evita atribuir al reservorio la atenuación de banda del procesamiento. Se guarda por separado la predicción física y la salida del operador de observación.

Hampel se utiliza para despikear mediciones. El modelo determinista no contiene el ruido ni los artefactos de bomba; se comprueba al final que aplicarle Hampel también no cambia materialmente su salida filtrada. El balance de masa se verifica sobre el reservorio físico, no sobre una señal suavizada de observación.

In [ ]:
CHEM_CONTEXT=[]
for context in ('reference_20_45','strong_30_60','raw_1min'):
    V.configure_temperature(context);V.prepare_all()
    for case in ('theta_full','theta_defaults'):CHEM_CONTEXT.append(V.chemistry_v3(case))
    print('Química integrada:',context,flush=True)
CHEM_METRICS=pd.concat(CHEM_CONTEXT,ignore_index=True)
display(CHEM_METRICS.groupby(['temperature','theta','channel']).RMSE.mean().unstack('channel').round(4))
V.set_context('reference_20_45')
display(V.initials)
display(V.biomass[['lab','sample','t_h','X_viable_g_l']].round(5))
CHEM_METRICS.to_csv(OUT/'chemistry_metrics.csv',index=False)

In [ ]:
for cohort in (LABS[3:],LABS[:3]):
    fig,axs=plt.subplots(3,4,figsize=(18,10),sharex=True,sharey='col')
    for i,lab in enumerate(cohort):
        for j,(channel,column,mult,unit) in enumerate([('N','YAN_mg_l',1000.,'mg N/L'),
            ('X','X_viable_g_l',1.,'g/L'),('GF','GF_g_l',1.,'g/L'),('E','E_g_l',1.,'g/L')]):
            ax=axs[i,j]
            for case,color,style in [('theta_full','#0072B2','-'),('theta_defaults','#D55E00','--')]:
                cr=V.cores[case,lab];keep=cr.index<=V.horizons[lab]
                ax.plot(cr.index[keep],cr[channel][keep]*mult,c=color,ls=style,label=case+' · T filtrada')
            previous=V.previous_chemistry.query('theta == "theta_full" and lab == @lab')
            ax.plot(previous.t_h,previous[channel]*mult,c='0.5',ls=':',lw=1.,label='theta_full v2 · T anterior')
            if lab in LABS[3:]:
                pts=V.chem[V.chem.lab.eq(lab)&~V.chem.is_NUT][['t_h',column]].dropna()
                ax.scatter(pts.t_h,pts[column],c='black',s=16,label='Muestra química/biomasa')
            elif channel=='X':
                pts=V.biomass[V.biomass.lab.eq(lab)]
                ax.scatter(pts.t_h,pts.X_viable_g_l,c='black',s=18,label='Oculyze viable')
            if channel=='GF':
                pts=V.density[V.density.lab.eq(lab)]
                ax.scatter(pts.t_h,pts.GF_density_g_l,c='#009E73',marker='s',s=18,label='G+F por densidad (aproximación)')
            for t,dose in V.batches[lab].pulses['N']:ax.axvline(t,c='deeppink',ls=':',lw=.8)
            ax.set(title=f'{lab} · {channel}',ylabel=unit,xlabel='Tiempo desde inoculación [h]',xlim=(0.,max(V.horizons[l] for l in cohort)))
            ax.grid(alpha=.2)
    handles={}
    for ax in axs[0]:
        hh,ll=ax.get_legend_handles_labels()
        for h,l in zip(hh,ll):handles.setdefault(l,h)
    fig.legend(list(handles.values()),list(handles),loc='upper center',ncol=3,fontsize=9)
    show_v3(fig,legend=True)

## 5. Ajuste principal Z1/Z2 y separación de calibración y transferencia

Z1 estima sólo `matrix_gain`, con k y saturación heredados. Z2 estima k, saturación y ganancia dentro de los mismos límites de v2, con tres inicializaciones. Se conservan el rendimiento CO₂/E y la conversión SCCM→g/L/h. La fuente no utiliza CO₂ observado para activar la fermentación.

Cada LAB pesa igual mediante residuos divididos por √n. Para y≥0.05 g/L/h se usa ŷ−y; por debajo del límite se penaliza max(ŷ−0.05,0). **Todos los RMSE publicados son sin censura.** Los puntos filtrados están correlacionados; la razón singular del Jacobiano es un diagnóstico local, no un intervalo de confianza ni evidencia de muestras independientes.

Se presentan: ajuste conjunto de seis LAB (descriptivo), entrenamiento en LAB019–021 y predicción de LAB016–018 (transferencia principal), y entrenamiento inverso (comprobación adicional). No se selecciona la intensidad del filtro usando el holdout. Los parámetros v2 se reevalúan bajo las mismas entradas, operador y observaciones v3 para una comparación justa; sus antiguos RMSE horarios no se comparan directamente con los nuevos RMSE de diez minutos.

In [ ]:
GROUPS={'all_six':LABS,'train_019_021':LABS[3:],'train_016_018':LABS[:3]}
for case in ('theta_full','theta_defaults'):
    for label,training in GROUPS.items():V.fit_layer(case,training,label)
    pd.DataFrame(V.fit_table).to_csv(OUT/'fit_parameters.csv',index=False)
FIT_TABLE=pd.DataFrame(V.fit_table)
display(FIT_TABLE.round(6))
MAIN_METRICS=[];MAIN_CURVES=[];PARAMETER_CONTROL=[]
for case in ('theta_full','theta_defaults'):
    for training in GROUPS:
        for stage in ('Z1','Z2'):
            key=('reference_20_45',PRIMARY,'v2',case,training,stage)
            params=V.v3_fits[key]
            met=V.metrics_v3(case,params,LABS).assign(training=training,stage=stage,source='v3_fit')
            MAIN_METRICS.append(met)
            for lab in LABS:
                cr=V.curve_v3(case,lab,params)
                MAIN_CURVES.append(cr[cr.t_h<=V.horizons[lab]+1e-8].assign(lab=lab,theta=case,training=training,stage=stage))
        old=V.previous_parameters.query('theta == @case and training == @training and stage == "Z2"').iloc[0]
        params={**V.old,**{p:float(old[p]) for p in ('kCO2_release_h','CO2sat_scale','matrix_gain')}}
        control=V.metrics_v3(case,params,LABS).assign(training=training,stage='Z2',source='v2_parameters_on_v3_data')
        PARAMETER_CONTROL.append(control)
METRICS=pd.concat(MAIN_METRICS+PARAMETER_CONTROL,ignore_index=True)
CURVES=pd.concat(MAIN_CURVES,ignore_index=True)
METRICS['role']=['calibration' if row.training=='all_six' or row.lab in GROUPS[row.training] else 'transfer' for row in METRICS.itertuples()]
display(METRICS.query('window == "all"').groupby(['theta','training','stage','source','role']).RMSE.mean().to_frame().round(5))
METRICS.to_csv(OUT/'metrics.csv',index=False)
CURVES.to_csv(OUT/'gas_trajectories.csv',index=False)

In [ ]:
for case in ('theta_full','theta_defaults'):
    for kind in ('joint','transfer'):
        fig,axs=plt.subplots(2,3,figsize=(17,8),sharex='col')
        for ax,lab in zip(axs.flat,LABS):
            training='all_six' if kind=='joint' else 'train_019_021' if lab in LABS[:3] else 'train_016_018'
            dense=V.dense[lab];keep=dense.time_h.between(0.,V.horizons[lab])
            ax.plot(dense.time_h[keep],np.where(dense.valid_for_fit,dense[PRIMARY],np.nan)[keep],c='0.65',lw=.8,label='CO₂ offline · señal densa válida')
            obs=V.observations[PRIMARY,lab]
            ax.scatter(obs.t_h,obs.y,c='black',s=5,label='Medianas 10 min usadas',zorder=5)
            for stage,color,style in [('Z1','#E69F00','--'),('Z2','#0072B2','-')]:
                curve=CURVES.query('theta == @case and lab == @lab and training == @training and stage == @stage')
                ax.plot(curve.t_h,curve.CO2_observed_operator,c=color,ls=style,lw=1.6,label=stage+' · v3')
            for t,dose in V.batches[lab].pulses['N']:ax.axvline(t,c='deeppink',ls=':',lw=.8)
            ax.set(title=f'{lab} · '+('ajuste conjunto' if kind=='joint' else f'entrenado en {training[6:]}'),
                   xlabel='Tiempo desde inoculación [h]',ylabel='g CO₂/L/h',xlim=(0.,max(V.horizons.values())))
            ax.grid(alpha=.2)
        fig.legend(*axs[0,0].get_legend_handles_labels(),loc='upper center',ncol=4,fontsize=9)
        fig.suptitle(case+' · '+kind,y=1.015,fontsize=14)
        show_v3(fig,legend=True)

## 6. Sensibilidad al filtrado de CO₂ y temperatura

Se repite el entrenamiento LAB019–021 con τ=20 y 40 min manteniendo la química principal. Se repite después con T 30/60 y τ=30. Se publican errores sobre cada señal procesada y, por separado, sobre la misma mediana de diez minutos del raw despikeado, comparada con la predicción física sin Butterworth. Así un filtro más fuerte no gana automáticamente por haber borrado variación real de su objetivo.

El caso principal sigue siendo 30 min y T 20/45, fijado antes de estimar parámetros. Las diferencias de transferencia de esta tabla son exploratorias. El suavizado no demuestra por sí solo que una oscilación atenuada sea ruido: puede contener respuesta al control térmico. Tampoco reestima la dependencia Arrhenius de la cinética.

In [ ]:
SENSITIVITY=[]
for context,filtering in [('reference_20_45','offline_20min'),('reference_20_45',PRIMARY),
                           ('reference_20_45','offline_40min'),('strong_30_60',PRIMARY)]:
    V.set_context(context)
    for case in ('theta_full','theta_defaults'):
        key=(context,filtering,'v2',case,'train_019_021','Z2')
        if key not in V.v3_fits:V.fit_layer(case,LABS[3:],'train_019_021',filtering=filtering,starts=2)
        params=V.v3_fits[key]
        for common in (False,True):
            met=V.metrics_v3(case,params,LABS,filtering=filtering,common_target=common)
            SENSITIVITY.append(met.assign(temperature=context,filtering=filtering))
SENSITIVITY=pd.concat(SENSITIVITY,ignore_index=True)
SENSITIVITY['cohort']=np.where(SENSITIVITY.lab.isin(LABS[:3]),'transfer_016_018','train_019_021')
display(SENSITIVITY.query('window == "all"').groupby(['temperature','filtering','theta','target','cohort']).RMSE.mean().to_frame().round(5))
SENSITIVITY.to_csv(OUT/'filter_sensitivity.csv',index=False)
V.set_context('reference_20_45')

## 7. Comparación adicional: mezcla positiva y ganancia 1.025

Se conserva el interés del seguimiento anterior: comparar la fuente original con una mezcla M·[(1−r)q₀+b·r·qₙ] que evita la resta final de la trayectoria sin nutrición. Es una modificación estructural local, **no forma parte de la recalibración principal Z1/Z2 de v3**. Se prueban `theta_full` y el mismo theta con sN=18; cada uno se ajusta sólo con LAB019–021. Además se muestra el cambio exclusivo a matrix_gain=1.025 sin reajuste. El operador offline y las entradas son los mismos en todos los casos.

In [ ]:
EXTRA_FITS={};EXTRA_METRICS=[];EXTRA_CURVES=[]
case='theta_full_sN18'
V.prepare(case,{**V.thetas['theta_full'],'sN':18.},LABS)
for case in ('theta_full','theta_full_sN18'):
    z1,z2=V.fit_layer(case,LABS[3:],'train_019_021',mode='positive_mix',starts=3)
    for label,params in [('positive_mix',z2),('positive_mix_gain_1p025',{**z2,'matrix_gain':1.025})]:
        EXTRA_FITS[case,label]=params
        EXTRA_METRICS.append(V.metrics_v3(case,params,LABS,mode='positive_mix').assign(case=case,scenario=label))
        for lab in LABS:
            curve=V.curve_v3(case,lab,params,mode='positive_mix')
            EXTRA_CURVES.append(curve[curve.t_h<=V.horizons[lab]].assign(lab=lab,case=case,scenario=label))
EXTRA_METRICS=pd.concat(EXTRA_METRICS,ignore_index=True)
EXTRA_CURVES=pd.concat(EXTRA_CURVES,ignore_index=True)
display(EXTRA_METRICS.query('window == "all"').assign(cohort=lambda d:np.where(d.lab.isin(LABS[:3]),'transfer_016_018','train_019_021')).groupby(['case','scenario','cohort']).RMSE.mean().to_frame().round(5))
fig,axs=plt.subplots(2,3,figsize=(17,8),sharex='col')
for ax,lab in zip(axs.flat,LABS):
    obs=V.observations[PRIMARY,lab];ax.scatter(obs.t_h,obs.y,c='black',s=5,label='CO₂ offline · 10 min')
    base=CURVES.query('theta == "theta_full" and training == "train_019_021" and stage == "Z2" and lab == @lab')
    ax.plot(base.t_h,base.CO2_observed_operator,c='#0072B2',lw=1.6,label='v3 fuente original Z2')
    for case,color in [('theta_full','#CC79A7'),('theta_full_sN18','#009E73')]:
        for scenario,style in [('positive_mix','--'),('positive_mix_gain_1p025','-')]:
            curve=EXTRA_CURVES.query('case == @case and scenario == @scenario and lab == @lab')
            ax.plot(curve.t_h,curve.CO2_observed_operator,c=color,ls=style,lw=1.4,label=case+' · '+scenario)
    ax.set(title=lab+' · '+('transferencia' if lab in LABS[:3] else 'entrenamiento'),xlabel='Tiempo desde inoculación [h]',ylabel='g CO₂/L/h');ax.grid(alpha=.2)
fig.legend(*axs[0,0].get_legend_handles_labels(),loc='upper center',ncol=3,fontsize=8)
show_v3(fig,legend=True)
EXTRA_METRICS.to_csv(OUT/'positive_mix_metrics.csv',index=False)
EXTRA_CURVES.to_csv(OUT/'positive_mix_curves.csv',index=False)

## 8. Verificación numérica y alcance

Se comprueban balances físicos, positividad, inventario N con pulsos exactos, convergencia del reservorio y efecto del operador de observación. El Hampel se aplica a las predicciones finales como auditoría: se compara con el operador Butterworth usado durante la optimización para verificar si la ausencia de outliers del modelo hace válida esa simplificación. Se exporta su diferencia, sin confundirla con error químico.

No se interpretan las nuevas ganancias como rendimientos químicos. La cancelación de fuentes de v2, las condiciones iniciales inciertas, las muestras de NUT sin hora confirmada y los errores de química de LAB021 siguen siendo limitaciones posibles. El filtrado centrado de temperatura modifica la entrada de un modelo no lineal; la cinética evaluada a T suavizada no equivale exactamente al promedio de la cinética a T instantánea. Por eso se publica la sensibilidad térmica y no se atribuye automáticamente una mejora al filtro.

In [ ]:
NUMERICAL=[]
for case in ('theta_full','theta_defaults'):
    params=V.v3_fits[('reference_20_45',PRIMARY,'v2',case,'train_019_021','Z2')]
    for lab in LABS:
        cache=V.source_cache(case,lab,params,'v2')
        observed,physical,cd,qout=V.observed_prediction(case,lab,params)
        balance=float(np.max(abs(np.diff(cd)-cache['dt']*(cache['source'][:-1]-qout[1:]))))
        core=V.cores[case,lab];theta=V.theta_by_case[case]
        dose=np.array([sum(d for t,d in cache['events'] if t<ti) for ti in cache['time']])
        inventory=core.N+theta['qN']/theta['mu0']*(core.X+core.Xd)-dose
        n_error=float(np.max(abs(inventory-inventory.iloc[0])))
        fine=V.observed_prediction(case,lab,params,max_step=1./48.)[0]
        obs=V.observations[PRIMARY,lab];t=V.dense[lab].time_h.to_numpy()
        coarse_sample=np.interp(obs.t_h,t,observed);fine_sample=np.interp(obs.t_h,t,fine)
        dt,blocks=V.filter_blocks[lab]
        native=np.interp(t,cache['time'],physical,left=0.)
        full_hampel=np.full(len(t),np.nan)
        for a,b in blocks:
            clean,flag=centered_hampel(native[a:b]/FACTOR,dt)
            full_hampel[a:b]=np.maximum(zero_phase(clean*FACTOR,FILTERS[PRIMARY],dt),0.)
        hampel_error=float(np.max(abs(np.interp(obs.t_h,t,full_hampel)-coarse_sample)))
        assert balance<1e-10 and n_error<2e-6
        assert min(np.min(cd),np.min(physical),np.min(cache['source']))>=-1e-12
        assert np.max(abs(coarse_sample-fine_sample))<.025
        assert hampel_error<.001
        NUMERICAL.append(dict(theta=case,lab=lab,balance=balance,nitrogen_inventory=n_error,
            reservoir_refinement_max=float(np.max(abs(coarse_sample-fine_sample))),
            model_hampel_operator_max=hampel_error))
NUMERICAL=pd.DataFrame(NUMERICAL);display(NUMERICAL)
assert pd.DataFrame(V.fit_table).success.all()
assert V.verify_protected()>0
pd.DataFrame(V.fit_table).to_csv(OUT/'fit_parameters.csv',index=False)
pd.DataFrame(V.fit_runs).to_csv(OUT/'optimizer_runs.csv',index=False)
NUMERICAL.to_csv(OUT/'numerical_checks.csv',index=False)
OBS_EXPORT=pd.concat([table.assign(filtering=name) for (name,lab),table in V.observations.items()],ignore_index=True)
OBS_EXPORT.to_csv(OUT/'observations_10min.csv',index=False)
CHEM_EXPORT=[]
for context in V.saved_core_contexts:
    V.set_context(context)
    for (case,lab),core in V.cores.items():
        CHEM_EXPORT.append(core[core.index<=V.horizons[lab]].rename_axis('t_h').reset_index().assign(lab=lab,theta=case,temperature=context))
pd.concat(CHEM_EXPORT,ignore_index=True).to_csv(OUT/'chemistry_trajectories.csv',index=False)
V.set_context('reference_20_45')
manifest=dict(created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    source_notebook_sha256=sha256(ND/'co2_lab016_021_recalibration_v3.source.ipynb'),
    support_module_sha256=sha256(ND/'_co2_recalibration_v3.py'),
    gui_parameters=V.gui,primary_filter=PRIMARY,filter_tau_seconds=FILTERS,
    hampel='centered odd window; exact same-window median/MAD; 120 s, k=3.5, floor=0.5 SCCM',
    butterworth='order 3, forward-backward SOS, odd padding six tau; amplitude response squared',
    temperature_primary='1min median -> centered 20min median -> centered 45min mean',
    temperature_sensitivity='centered 30min median -> centered 60min mean',
    observation_interval_minutes=10,observations='median per window, center-labelled',
    model_temperature_is_same_filtered_temperature_as_plots=True,
    model_observation_operator='same native-grid zero-phase Butterworth; model Hampel checked numerically',
    clock_timezone=LOCAL_ZONE,computation_timezone='UTC',clock_change_audit=V.clock_audit,
    exclusion_margin_hours=0,nutrition_excluded=False,pump_bridge_applied=False,
    main_chemistry_fixed=V.thetas,initials=V.initials.reset_index().to_dict('records'),
    coverage=V.coverage.to_dict('records'),main_fit_modes=['Z1','Z2'],
    main_training_labs=list(LABS[3:]),main_transfer_labs=list(LABS[:3]),
    joint_six_lab_fit_is_descriptive=True,filter_chosen_using_holdout=False,
    protected_input_sha256=V.protected,figures='inline only',figure_count=FIGURE_COUNT,
    previous_outputs_modified=False,elapsed_seconds=time.perf_counter()-START)
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
display(Markdown('### Resumen calculado de v3 corregida'))
display(Markdown('**Corrección temporal y de soporte:** el supuesto hueco de una hora era el cambio de hora de Chile. Entre esos registros transcurren 14 segundos. Se quitaron las exclusiones por bomba, los 90 minutos posteriores y los márgenes de cuatro horas. Las muestras de nutrición y sus respuestas entran ahora en el ajuste y en los errores. Las conclusiones y métricas anteriores de v3 quedan sustituidas por esta ejecución; no son comparables punto a punto porque cambió el soporte.'))
for case in ('theta_full','theta_defaults'):
    current=METRICS.query('theta == @case and training == "train_019_021" and stage == "Z2" and source == "v3_fit" and window == "all" and role == "transfer"').RMSE.mean()
    control=METRICS.query('theta == @case and training == "train_019_021" and source == "v2_parameters_on_v3_data" and window == "all" and role == "transfer"').RMSE.mean()
    display(Markdown(f'- **{case}, transferencia LAB016–018:** RMSE medio v3={current:.4f} g/L/h; parámetros v2 reevaluados sobre estos mismos datos={control:.4f}.'))
display(Markdown(f'- **Datos:** {len(V.obs)} medianas válidas de diez minutos, frente a {len(V.v2_obs)} medianas horarias anteriores. Las máscaras y los horizontes también cambiaron; el cociente no representa sólo la frecuencia de muestreo.\n- **Cobertura:** LAB020 se extiende a {V.horizons["LAB020"]:.2f} h; todas las últimas muestras químicas quedan cubiertas. LAB016: final censurado={bool(V.coverage.set_index("lab").loc["LAB016","right_censored"])}, corte={V.horizons["LAB016"]:.2f} h reales.\n- **Filtros:** configuración principal fijada en τ=30 min y T 20/45. Las tablas de sensibilidad conservan los resultados alternativos; no prueban una calibración química definitiva.'))
display(Markdown('**Interpretación:** recuperar el soporte y corregir el reloj hace válida la comparación temporal, pero no resuelve por sí solo las limitaciones del modelo. La fuente original con theta_full todavía anticipa el inicio y cae demasiado pronto al final en LAB016–018; también subestima el máximo de LAB021. La recalibración principal no mejora la transferencia frente a los parámetros v2 reevaluados sobre estas mismas observaciones.'))
for extra_case in ('theta_full','theta_full_sN18'):
    for scenario in ('positive_mix','positive_mix_gain_1p025'):
        subset=EXTRA_METRICS.query('case == @extra_case and scenario == @scenario and window == "all"')
        holdout=subset[subset.lab.isin(LABS[:3])].RMSE.mean()
        training=subset[subset.lab.isin(LABS[3:])].RMSE.mean()
        display(Markdown(f'- **{extra_case} / {scenario}:** RMSE medio entrenamiento={training:.4f}; transferencia={holdout:.4f} g/L/h.'))
display(Markdown('Las mezclas positivas evitan la cancelación de la fuente final y la ganancia fija modifica su amplitud, pero persisten discrepancias de inicio y de máximos. Se presentan como alternativas exploratorias; no se selecciona una ganancia o un filtro a partir del holdout ni se interpreta una caída del RMSE respecto a la primera v3 como mejora sobre el mismo conjunto de puntos.'))
print('Archivos previos protegidos:',V.verify_protected(),'; figuras integradas:',FIGURE_COUNT,'; tiempo:',round(time.perf_counter()-START,1),'s')